# Границы файнтюнинга: обучение фактов с контролем переобучения

Обучение LoRA на 22 фактах (7 синтетических + 4 реальных 2026 года) **10 эпох**,
valid-лосс каждые 5 шагов, чекпоинт каждую эпоху. Цель — увидеть, где train
падает, а valid перестаёт (переобучение), и получить facts-модель для eval.

Загрузите ноутбук в Colab, включите T4, затем выполняйте ячейки по порядку.

In [ ]:
!pip -q install -U "transformers>=4.56" "peft>=0.13" accelerate matplotlib
!pip -q uninstall -y torchao
# После этой ячейки: Среда выполнения → Перезапустить сеанс, затем выполнять со следующей.

In [ ]:
import glob
import os
import subprocess

from google.colab import files

os.chdir('/content')
project = '/content/finetuning/boundaries'
train = f'{project}/data/train.jsonl'

if not os.path.exists(train):
    print('Выберите файл boundaries.zip')
    files.upload()
    zips = sorted(glob.glob('/content/boundaries*.zip'))
    if not zips:
        raise RuntimeError('zip не найден в /content')
    subprocess.run(['unzip', '-qo', zips[-1], '-d', '/content'], check=True)

os.chdir(project)
print('проект:', os.getcwd())
print('train:', os.path.exists('data/train.jsonl'), '| valid:', os.path.exists('data/valid.jsonl'))

In [ ]:
# Проверить данные: контаминация перефразировок eval в train/valid должна быть 0.
!python data/build_facts.py

In [ ]:
# Обучение: 10 эпох, valid каждые 5 шагов, чекпоинт каждую эпоху.
# run_params.json и loss.png запишутся в results/ проекта.
!python training/train_facts.py \
  --epochs 10 --eval-steps 5 --save-strategy epoch \
  --out /content/adapter_facts \
  --run-dir /content/run_facts

In [ ]:
# График train vs valid — по нему видно расхождение (переобучение).
import json
from IPython.display import Image, display

run = json.load(open('results/run_params.json'))
print('train loss', run['loss_first'], '->', run['loss_last'])
print('valid loss first', run['eval_loss_first'], '| min на шаге', run['eval_loss_min_step'], '| last', run['eval_loss_last'])
display(Image('results/loss.png'))
print('чекпоинты (в /content/run_facts):', sorted(os.listdir('/content/run_facts')))

## Eval дообученной модели прямо здесь (без GGUF)

`eval_hf.py` гоняет адаптер через transformers/peft и пишет тот же JSON, что
Ollama-скрипты. Judge (покрытие фактов) использует DeepSeek — введите ключ,
либо оставьте пустым и получите только детерминированные `keys_ok`.

Дополнительно можно прогнать **лучший чекпоинт** (`checkpoint_best`) и сравнить
с переобученным финальным адаптером.

In [ ]:
import os
from getpass import getpass

if not os.environ.get('DEEPSEEK_API_KEY'):
    os.environ['DEEPSEEK_API_KEY'] = getpass('DEEPSEEK_API_KEY (нужен для judge; пусто — без judge): ')
no_judge = '' if os.environ.get('DEEPSEEK_API_KEY') else '--no-judge'

# переобученный финальный адаптер (30 шагов)
!python scripts/eval_hf.py --task facts --adapter /content/adapter_facts {no_judge} --out results/facts/tuned.json
!python scripts/eval_hf.py --task ood --adapter /content/adapter_facts --out results/ood/tuned.json

## Упаковать результат

Архив повторяет пути репозитория, поэтому локально восстанавливается командой
`unzip -o boundaries_results.zip -d <repo>`: результаты (включая `run_params.json`,
`loss.png`, `facts/tuned.json`, `ood/tuned.json`) и **два адаптера** —
`adapter_facts` (финальный, переобученный) и `checkpoint_best` (лучший по valid,
без служебных файлов optimizer/scheduler).

In [ ]:
import os
import shutil

artifacts = '/content/finetuning/artifacts/boundaries'
os.makedirs(artifacts, exist_ok=True)

# Финальный (переобученный) адаптер — целиком.
dst_final = f'{artifacts}/adapter_facts'
if os.path.exists(dst_final):
    shutil.rmtree(dst_final)
shutil.copytree('/content/adapter_facts', dst_final)

# Лучший чекпоинт (мин. valid): только файлы адаптера, без optimizer/scheduler/rng.
run = json.load(open('results/run_params.json'))
best_step = run['eval_loss_min_step'][0]
src_best = f'/content/run_facts/checkpoint-{best_step}'
dst_best = f'{artifacts}/checkpoint_best'
if os.path.exists(dst_best):
    shutil.rmtree(dst_best)
os.makedirs(dst_best, exist_ok=True)
for name in ['adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']:
    shutil.copy2(f'{src_best}/{name}', f'{dst_best}/{name}')
print('best checkpoint:', best_step, '->', dst_best)

!cd /content && rm -f boundaries_results.zip
!cd /content && zip -qr boundaries_results.zip \
  finetuning/boundaries/results \
  finetuning/artifacts/boundaries/adapter_facts \
  finetuning/artifacts/boundaries/checkpoint_best

from google.colab import files
files.download('/content/boundaries_results.zip')